In [19]:
import sys
import subprocess

In [16]:
# Auto install required packages if not already installed
import sys
import subprocess

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# List of required packages
packages = [
    "pandas",
    "numpy",
    "scikit-learn",
    "tensorflow",
    "joblib",
    "matplotlab"
]

for package in packages:
    try:
        __import__(package)
    except ImportError:
        install(package)


CalledProcessError: Command '['D:\\AB\\Barclay\\mlmodel\\M3\\apifaultcause\\Scripts\\python.exe', '-m', 'pip', 'install', 'matplotlab']' returned non-zero exit status 1.

In [20]:
sys.execute("pip install jupyter")


AttributeError: module 'sys' has no attribute 'execute'

In [13]:
# API Crash Prediction Notebook

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
import joblib
import os

# 1. Data Loading
# Function to load and process log files
def load_logs(file_path):
    all_data = []
    
    with open(file_path, 'r') as f:
        for line in f:
            try:
                log_entry = json.loads(line.strip())
                all_data.append(log_entry)
            except json.JSONDecodeError:
                print(f"Skipping invalid JSON line in {file_path}")
    
    return all_data

# Load log data - assuming the file is in the same directory as the notebook
log_file = 'api_log.json'
logs = load_logs(log_file)

# Convert logs to DataFrame
df = pd.DataFrame(logs)

# Display first few rows to understand the data
print("Data Overview:")
df.head()

# 2. Exploratory Data Analysis
# Check basic statistics and information
print("\nDataFrame Info:")
df.info()

print("\nBasic Statistics:")
df.describe()

# Plot response time distribution
plt.figure(figsize=(10, 6))
sns.histplot(df['response_time_ms'], kde=True)
plt.title('Response Time Distribution')
plt.xlabel('Response Time (ms)')
plt.ylabel('Frequency')
plt.show()

# 3. Feature Engineering
# Extract features that might indicate system stress
df_features = pd.DataFrame()

# Response time is a key indicator of system stress
df_features['response_time_ms'] = df['response_time_ms']

# Extract hour from timestamp for time-based patterns
df_features['hour'] = pd.to_datetime(df['timestamp']).dt.hour

# Method type (GET, POST, PUT, etc.)
df_features['is_write_operation'] = df['method'].apply(lambda x: 1 if x in ['POST', 'PUT', 'DELETE', 'PATCH'] else 0)

# Status code - convert to binary (success/error)
df_features['is_error'] = (df['status_code'] >= 400).astype(int)

# Request body size might impact performance
df_features['request_size'] = df.apply(
    lambda row: len(str(row['request_body'])) if isinstance(row['request_body'], dict) else 0, 
    axis=1
)

# Environment feature (encoded)
df_features['is_onprem'] = (df['environment'] == 'on-prem').astype(int)

# 4. Define Target Variable
# Let's define "about to crash" as response times in the top 15%
threshold = df['response_time_ms'].quantile(0.85)
df_features['about_to_crash'] = (df['response_time_ms'] > threshold).astype(int)

print(f"\nPercentage of records indicating potential crash: {df_features['about_to_crash'].mean() * 100:.2f}%")

# Visualize feature correlations
plt.figure(figsize=(10, 8))
sns.heatmap(df_features.corr(), annot=True, cmap='coolwarm')
plt.title('Feature Correlation Matrix')
plt.show()

# 5. Prepare data for modeling
# Remove response time from features as it's used for the target (to avoid data leakage)
X = df_features.drop(['response_time_ms', 'about_to_crash'], axis=1)
y = df_features['about_to_crash']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. Build neural network model
model = Sequential([
    Dense(12, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.2),
    Dense(6, activation='relu'),
    Dropout(0.1),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("\nModel Summary:")
model.summary()

# 7. Train the model
history = model.fit(
    X_train_scaled, y_train,
    epochs=30,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

# 8. Evaluate the model
# Plot training history
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()

# Test set evaluation
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test)
print(f"\nTest accuracy: {test_accuracy:.4f}")

# Generate predictions
y_pred_prob = model.predict(X_test_scaled)
y_pred = (y_pred_prob > 0.5).astype(int)

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Confusion matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# 9. Save the model and scaler
model.save('api_crash_model.h5')
joblib.dump(scaler, 'api_crash_scaler.save')
print("\nModel and scaler saved successfully.")

# 10. Create prediction function for later use in Flask app
def predict_api_crash(new_data):
    """
    Predict if API is about to crash based on new log data
    
    Parameters:
    new_data: A dictionary with log data fields
    
    Returns:
    Probability of API crash and binary prediction
    """
    # Create features from new data
    features = {
        'hour': pd.to_datetime(new_data.get('timestamp', pd.Timestamp.now())).hour,
        'is_write_operation': 1 if new_data.get('method', 'GET') in ['POST', 'PUT', 'DELETE', 'PATCH'] else 0,
        'is_error': 1 if new_data.get('status_code', 200) >= 400 else 0,
        'request_size': len(str(new_data.get('request_body', {}))),
        'is_onprem': 1 if new_data.get('environment', '') == 'on-prem' else 0
    }
    
    # Convert to DataFrame
    df_pred = pd.DataFrame([features])
    
    # Scale features
    scaled_features = scaler.transform(df_pred)
    
    # Make prediction
    crash_probability = model.predict(scaled_features)[0][0]
    prediction = 'API may crash soon' if crash_probability > 0.5 else 'API stable'
    
    return crash_probability, prediction

# 11. Test prediction function with an example
example_log = {
    "timestamp": "2025-04-26 18:22:14.934681",
    "method": "PUT",
    "endpoint": "/update",
    "status_code": 200,
    "request_body": {"name": "Test", "value": 42},
    "environment": "on-prem",
    "response_time_ms": 800
}

crash_probability, prediction = predict_api_crash(example_log)
print(f"\nExample Prediction:")
print(f"Probability of API crash: {crash_probability:.4f}")
print(f"Prediction: {prediction}")

# 12. Feature Importance Analysis (using a simple approach for neural networks)
# We'll use permutation importance by shuffling each feature
def calculate_feature_importance(model, X, y, n_repeats=5):
    from sklearn.inspection import permutation_importance
    
    # Get baseline score
    y_pred = model.predict(X)
    baseline_score = np.mean((y_pred.flatten() > 0.5) == y)
    
    importances = {}
    feature_names = X.columns
    
    for i, feature in enumerate(feature_names):
        scores = []
        for _ in range(n_repeats):
            # Create a copy of X
            X_permuted = X.copy()
            # Shuffle the feature
            X_permuted[feature] = np.random.permutation(X_permuted[feature].values)
            # Scale the data
            X_permuted_scaled = scaler.transform(X_permuted)
            # Get predictions
            y_pred_permuted = model.predict(X_permuted_scaled)
            # Calculate score
            score = np.mean((y_pred_permuted.flatten() > 0.5) == y)
            # Importance is the decrease in score
            importance = baseline_score - score
            scores.append(importance)
        
        importances[feature] = np.mean(scores)
    
    return importances

# Calculate feature importance
importances = calculate_feature_importance(model, X_test, y_test)

# Plot feature importance
plt.figure(figsize=(10, 6))
features = list(importances.keys())
values = list(importances.values())
plt.barh(features, values)
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.tight_layout()
plt.show()

print("\nFeature Importance:")
for feature, importance in sorted(importances.items(), key=lambda x: x[1], reverse=True):
    print(f"{feature}: {importance:.4f}")

# 13. Root Cause Analysis
print("\nPotential Root Causes of API Crashes:")
# Identify patterns in the data that might indicate causes of crashes
high_response_times = df[df['response_time_ms'] > threshold]

# Check if certain hours have more crashes
hour_crashes = high_response_times['timestamp'].apply(lambda x: pd.to_datetime(x).hour).value_counts().sort_index()
print("\nHours with high crash probability:")
print(hour_crashes)

# Check if certain endpoints have more crashes
if 'endpoint' in df.columns:
    endpoint_crashes = high_response_times['endpoint'].value_counts()
    print("\nEndpoints with high crash probability:")
    print(endpoint_crashes.head())

# Check if certain methods have more crashes
method_crashes = high_response_times['method'].value_counts()
print("\nMethods with high crash probability:")
print(method_crashes)

# Check relationship between request size and crashes
plt.figure(figsize=(10, 6))
sns.boxplot(x='about_to_crash', y='request_size', data=df_features)
plt.title('Request Size vs Crash Probability')
plt.xlabel('API About to Crash (1=Yes, 0=No)')
plt.ylabel('Request Size')
plt.show()

print("\nBased on the analysis, the most likely causes of API crashes appear to be:")
print("- [Model will determine based on actual data]")
print("- [Model will determine based on actual data]")
print("- [Model will determine based on actual data]")

print("\nRecommendations to prevent crashes:")
print("- Monitor these indicators in real-time")
print("- Set up alerting when risk factors are detected")
print("- Consider load balancing or rate limiting during high-risk periods")

ModuleNotFoundError: No module named 'matplotlib'